In [ ]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

<bound method NDFrame.head of                                               messages
0    [{'role': 'user', 'content': 'intro'}, {'role'...
1    [{'role': 'user', 'content': 'Who is Varad?'},...
2    [{'role': 'user', 'content': 'Tell me about Va...
3    [{'role': 'user', 'content': 'What is Varad li...
4    [{'role': 'user', 'content': 'What makes Varad...
..                                                 ...
106  [{'role': 'user', 'content': 'Are you consciou...
107  [{'role': 'user', 'content': 'How was this AI ...
108  [{'role': 'user', 'content': 'What is Varad's ...
109  [{'role': 'user', 'content': 'Does Varad know ...
110  [{'role': 'user', 'content': 'What does Varad ...

[111 rows x 1 columns]>


In [ ]:
df['messages'][1][1]['content']

"Varad Bendale. 2nd year Information Technology at K J Somaiya Institute of Technology, Mumbai. 9.57 CGPA — yeah that's not a typo. He's the kind of person who learns Random Forest and immediately goes 'cool, let me rebuild this in C++ from scratch with zero libraries.' That tells you everything. Genuinely passionate about AI and ML, not the surface level stuff — the deep, messy, how-does-this-actually-work kind. Wants to make real impact. Currently making me exist, which is either genius or madness, jury's still out."

In [36]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())

user_messege = []
asistant_reply = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    asistant_reply.append(df['messages'][i][1]['content'])

question_tokens  = []
target_tokens = []

for i in range(len(df['messages'])) :
    user_encods = enc.encode( "<|user|>"  + user_messege[i] ,  allowed_special= allowed)
    asis_encods = enc.encode( "<|assistant|>" +  asistant_reply[i] + "<|end|>"  , allowed_special= allowed )
    tokens = user_encods + asis_encods
    question = torch.tensor(tokens[:-1])
    question_tokens.append(question)
    target    = torch.tensor(tokens[1:])
    target_tokens.append(target)





In [ ]:
print(question_tokens)

In [17]:
g = torch.Generator()
g.manual_seed(42)

In [34]:
max_num = max(set(tokens))
print(max_num)

173781


In [45]:
vocab_size = len(set(tokens))
print(vocab_size)

107


In [48]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[tok.item()] for tok in question_tokens[i]])
    t_tensor = torch.tensor([stoi[tok.item()] for tok in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [50]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [58]:
dim = 100
W1 = torch.randn((vocab_size, dim), requires_grad=True)
W2 = torch.randn((dim, vocab_size), requires_grad=True)
xenc   = torch.nn.functional.one_hot(q_tokens, num_classes=vocab_size).float()
hidden = torch.tanh(xenc @ W1)
logits = hidden @ W2

In [57]:
loss = torch.nn.functional.cross_entropy(logits, t_tokens)
print(loss)

tensor(21.9560, grad_fn=<NllLossBackward0>)
